In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding_functions = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = Chroma(
    embedding_function=embedding_functions,
    collection_name="income_tax_collection",
    persist_directory="./income_tax_collection",
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph


# Node 끼리 State를 통해서 소통
# LangGraph에서 State = Graph 전체가 공유하는 데이터 저장소
# State 구조를 설계 하는 것이 LangGraph에서 제일 중요한 첫 단계
class AgentState(TypedDict):
    query: str
    context: List[Document]
    answer: str


graph_builder = StateGraph(AgentState)

In [ ]:
# retrieve node
def retrieve(state: AgentState):
    query = state["query"]
    docs = retriever.invoke(query)
    return {"context": docs}

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [ ]:
from langsmith import Client

client = Client()
generate_prompt = client.pull_prompt(
    "rlm/rag-prompt", dangerously_pull_public_prompt=True
)


# generate node
def generate(state: AgentState):
    context = state["context"]
    query = state["query"]
    rag_chain = generate_prompt | llm

    response = rag_chain.invoke({"question": query, "context": context})
    return {"answer": response}

In [ ]:
from langsmith import Client
from typing import Literal

client = Client()
doc_relevance_prompt = client.pull_prompt(
    "langchain-ai/rag-document-relevance",
    dangerously_pull_public_prompt=True,
)


# router function for conditional edge - determines next destination (generate or rewrite)
def check_doc_relevance(state: AgentState) -> Literal["generate", "rewrite"]:
    query = state["query"]
    context = state["context"]
    print(f"context == {context}")
    doc_relevance_chain = doc_relevance_prompt | llm

    response = doc_relevance_chain.invoke({"question": query, "documents": context})
    print(f"doc relevance repsonse: {response}")
    if response["Score"] == 1:
        return "generate"
    return "rewrite"

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

dictionary = ["사람과 관련된 표현 -> 거주자"]

rewrite_prompt = PromptTemplate.from_template(f"""
사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요
사전: {dictionary}
질문: {{query}}
""")


# rewrite node
def rewrite(state: AgentState):
    query = state["query"]
    rewrite_chain = rewrite_prompt | llm | StrOutputParser()

    response = rewrite_chain.invoke({"query": query})
    return {"query": response}

In [ ]:
# Node 등록

graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)
graph_builder.add_node("rewrite", rewrite)

In [ ]:
# Edge 연결
from langgraph.graph import START, END

graph_builder.add_edge(START, "retrieve")
# 뜻: check_doc_relevance 후에 점선이 나온다. 즉 check_doc_relevance 함수가 다음 목적지를 결정한다.
graph_builder.add_conditional_edges("retrieve", check_doc_relevance)
graph_builder.add_edge("rewrite", "retrieve")
graph_builder.add_edge("generate", END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
inital_state = {"query": "연봉 5천만원 세금"}
graph.invoke(inital_state)